In [38]:
import numpy as np
import pandas as pd

df = pd.read_excel("../data/raw/Online Retail.xlsx")

print (df.head())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [39]:
# ==================================================
# Convert DataFrame columns to NumPy arrays
# ==================================================

quantity_arr = df["Quantity"].to_numpy()
unitprice_arr = df["UnitPrice"].to_numpy()
customer_id_arr = df["CustomerID"].to_numpy()
invoice_arr = df["InvoiceNo"].to_numpy().astype(str)
stockcode_arr = df["StockCode"].to_numpy().astype(str)
country_arr = df["Country"].to_numpy()

print ("Dataset Loaded Successfully.")
print ("Total Records:", len(df))

Dataset Loaded Successfully.
Total Records: 541909


In [40]:
# ==================================================
#  DATA CLEANING USING NUMPY
# ==================================================

# ===== Identify valid quantities =====
valid_quantity_mask = quantity_arr > 0

# ===== Identify valid unit prices =====
valid_price_mask = unitprice_arr > 0

# ===== Identify cancelled invoices =====
cancelled_mask = np.char.startswith(invoice_arr, "C")

# ===== Combine all conditions =====
valid_mask = (
    valid_quantity_mask
    & valid_price_mask
    & ~cancelled_mask
)

# ===== Apply the same mask to the dataset =====
clean_df = df[valid_mask].copy()

# ==================================================
# CLEANING REPORT
# ==================================================

print("========== DATA CLEANING REPORT ==========")

print(f"Original Records : {len(df):,}")
print(f"Cleaned Records  : {len(clean_df):,}")
print(f"Removed Records  : {len(df) - len(clean_df):,}")

========== DATA CLEANING REPORT ==========
Original Records : 541,909
Cleaned Records  : 530,104
Removed Records  : 11,805


In [41]:
# ==================================================
# VERIFY CLEANED DATASET
# ==================================================

# 1. Verify Quantity
clean_quantity = clean_df["Quantity"].to_numpy()

invalid_quantity = np.sum(clean_quantity <= 0)

print("Invalid Quantities:", invalid_quantity)


# 2. Verify UnitPrice
clean_unitprice = clean_df["UnitPrice"].to_numpy()

invalid_unitprice = np.sum(clean_unitprice <= 0)

print("Invalid Unit Prices:", invalid_unitprice)


# 3. Verify Cancelled Invoices
clean_invoiceid = clean_df["InvoiceNo"].to_numpy().astype(str)

cancelled_mask = np.char.startswith(clean_invoiceid, "C")

cancelled_count = np.sum(cancelled_mask)

print("Cancelled Invoice Rows:", cancelled_count)

Invalid Quantities: 0
Invalid Unit Prices: 0
Cancelled Invoice Rows: 0


In [42]:
# ==========================================
# STEP 5: ORIGINAL VS CLEANED DATA
# ==========================================

print("Original Records:", len(df))
print("Cleaned Records:", len(clean_df))
print("Removed Records:", len(df) - len(clean_df))

Original Records: 541909
Cleaned Records: 530104
Removed Records: 11805


In [43]:
# ==================================================
# SAVE CLEANED DATA WITH CORRECT DATA TYPES
# ==================================================

np.savez(
    "../data/processed/cleaned_sales.npz",
    quantity=clean_df["Quantity"].to_numpy(),
    unitprice=clean_df["UnitPrice"].to_numpy(),
    invoice=clean_df["InvoiceNo"].to_numpy().astype(str),
    customer_id=clean_df["CustomerID"].to_numpy(),
    stockcode=clean_df["StockCode"].to_numpy().astype(str),
    country=clean_df["Country"].to_numpy().astype(str),
    invoice_data=clean_df["InvoiceDate"].to_numpy().astype("datetime64[ns]")
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [44]:
# ==================================================
# Verify saved NPZ file
# ==================================================

saved_data = np.load(
    "../data/processed/cleaned_sales.npz"
)

print("Saved Arrays:", saved_data.files)
print("Total Cleaned Records:", len(saved_data["quantity"]))
print("Invalid Quantities:", np.sum(saved_data["quantity"] <= 0))
print("Invalid Prices:", np.sum(saved_data["unitprice"] <= 0))
print(
    "Cancelled Invoices:",
    np.sum(np.char.startswith(saved_data["invoice"], "C"))
)

saved_data.close()

Saved Arrays: ['quantity', 'unitprice', 'invoice', 'customer_id', 'stockcode', 'country', 'invoice_data']
Total Cleaned Records: 530104
Invalid Quantities: 0
Invalid Prices: 0
Cancelled Invoices: 0
